# Live folder watching

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/electronmicroscopy/quantem.widget/blob/main/docs/tutorials/watch_folder.ipynb)

During a microscope session, results land in a folder one file at a time. The
viewers can **watch that folder** and grow in place - keep one widget open at
the scope and let new acquisitions appear, instead of re-running a notebook
after every file.

Where this earns its keep:

- **Survey acquisition** - `Show2D.from_folder` adds a panel per new HAADF as
  it lands, so you pick the next field of view from what you already have.
- **In-situ / time series** - `Show3D.from_folder` appends frames to a
  scrubbable stack while the experiment runs.
- **4D-STEM sessions** - `Show4DSTEM.from_folder` appends each ready
  `*_master.h5` to the dataset slider, deferring files that cannot yet be read.

```{tip}
Run this exact notebook with the Colab badge above, or [View or download this notebook on GitHub](https://github.com/electronmicroscopy/quantem.widget/blob/main/docs/tutorials/watch_folder.ipynb). For finished results, use [HTML and file export](widget_export) to export interactive HTML or share a trusted notebook with widget state. A standalone HTML export is a snapshot; filesystem watching requires a live Python kernel.
```

In [ ]:
import subprocess
import sys

try:
    import google.colab  # noqa: F401
except Exception:
    pass
else:
    from google.colab import output

    output.enable_custom_widget_manager()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/electronmicroscopy/quantem.widget.git"],
        check=True,
    )


In [ ]:
from quantem.widget.datasets import showfolder_gold

folder = showfolder_gold(verbose=False)  # real gold HAADF session

## Watch survey images with Show2D

`Show2D.from_folder` reads every readable image at full resolution and starts
background polling (`watch=True` is the default, `watch_interval=1.0`
seconds). Files added to the folder later become new panels in this same
widget; partially-written files are retried until stable. Here it opens the
real session's overview images:

In [ ]:
import pathlib
import shutil
import tempfile

from quantem.widget import Show2D

session = pathlib.Path(tempfile.mkdtemp(prefix="live_session_"))
for p in sorted(folder.glob("*overview 0[12]*")):
    shutil.copy(p, session / p.name)

viewer = Show2D.from_folder(session)
viewer

With the kernel running, drop another file into the folder and the widget adds
the panel by itself within a poll interval:

```python
shutil.copy(next(folder.glob("*overview 03*")), session)  # a new panel appears
```

On a live session you would point it at the acquisition folder and leave it
open:

```python
viewer = Show2D.from_folder("/data/session", pattern="*.emd")   # keeps growing
viewer = Show2D.from_folder("/data/session", watch=False)       # one-shot read
```

## Watch a growing stack with Show3D

For same-size frames (in-situ series, tilt series, or a denoiser writing
results), `Show3D.from_folder` plays a local folder as one stack and appends new
frames in place. A missing live folder is created automatically, including its
parent directories, so this cell can run before the microscope writes its first
frame:

```python
from quantem.widget import Show3D

stack = Show3D.from_folder(
    "/data/growth_run",
    file_types=["emd", "png", "tif", "tiff"],
    pattern="frame_*",
    watch_interval=1.0,
)
stack
```

Use `file_types` to ignore unrelated products in the same acquisition folder.
It accepts extensions with or without the leading dot and ignores case. When
`pattern` is also supplied, an arriving file must match both filters. Supported
inputs include EMD, TIFF, PNG, NumPy, DM3/DM4, JPEG, BMP, and GIF. Each file
must decode to a 2D gray or RGB image, and every frame in one Show3D stack must
have the same height, width, and gray/RGB channel layout. Incompatible or
partially written files remain pending instead of disrupting the live stack.

This watches your local acquisition folder; the data are not uploaded to a
remote service. A standalone HTML export is a snapshot of the frames already
loaded and cannot continue watching without its live Python kernel.

## Watch a 4D-STEM session with Show4DSTEM

For live scope folders, construct the watcher from the folder itself. Each
readable `*_master.h5` joins the same Dataset slider as a lazy slot; selecting
the new dataset performs the scientific load. Masters whose required linked
data are not readable yet remain retryable:

```python
from quantem.widget import Show4DSTEM

viewer = Show4DSTEM.from_folder(
    "/data/session",
    pattern="*_master.h5",
    watch_interval=2.0,
    det_bin=1,   # native detector sampling when memory allows
)
viewer
```

Use `det_bin=2` or `4` only for an explicitly labeled preview or memory-limited
watch session. See [Show4DSTEM live scope folders](../api/show4dstem.md#live-scope-folders)
for CUDA, Apple Silicon, paging, and precision choices.

## From the command line

The CLI writes a live ShowFolder notebook for image folders and a lazy
Show4DSTEM notebook for master folders. Use the direct Python APIs above when
the full-resolution Show2D/Show3D watcher itself is the workflow under test:

```bash
quantem show2d ./frames/ --watch     # ShowFolder + live all-image Show2D preview
quantem show3d ./frames/ --watch     # ShowFolder + live all-image Show3D preview
quantem show4dstem ./masters/        # lazy live-master Show4DSTEM
```

## Related pages

- [ShowFolder session browser](showfolder) - browse and star a session, or open both all-image previews
- [IO/GPU](io_gpu) - the loaders these watchers are built on
- [Show4DSTEM tutorial](show4dstem) - virtual detectors on the appended data